# NIHONGA — Phase 5: Pretrain the Bridges

Phase 4 verified an untrained residual bridge inside the pinned Qwen-Image 2.1 pipeline. This phase follows [PLAN.md](PLAN.md#phase-5--pretrain-the-bridges): fit the bridges to paired teacher block inputs and outputs while freezing inherited model weights. The initial experiment compares blocks 2, 4, 5 and 3 independently using the same bottleneck width and training budget. A successful initialization smoke test does not select a winning candidate or establish quality recovery.

## 1. Freeze the teacher-target capture pilot

We first select and save **20 TRAIN prompts and 10 VALIDATION prompts** from the frozen Phase 1 pilot splits. Each of the five capability groups contributes four training prompts (two general and two hard) and two validation prompts (one general and one hard). Selection uses a salted deterministic hash order and prefers distinct subdimensions within each group. Source metadata, prompt, seed, aspect ratio and the existing Phase 2 resolution mapping are preserved.

We check selected source IDs and normalized exact prompt matches against each other, INTERNAL TEST, the Phase 3 probe set, and both benchmark languages. Normalization uses Unicode NFKC, case folding and collapsed whitespace. Validation examples are never optimizer inputs. All source and implementation hashes are bound to the saved request.

The capture contract specifies the frozen pinned teacher, BF16, the original 40-step scheduler and three zero-based stage indices: **0, 20 and 39**. For each candidate layer we will save paired block input/output tensors from at most 256 image tokens per stage and up to 64 valid prefix tokens during first-step extraction. Token indices and roles, masks, actual timesteps and sigmas, and replay inputs must be saved. The same sampled token indices serve every candidate. Cached stages contain target-image tokens only; padded text is excluded.

This code cell creates the manifest and selected rows locally. It does not capture teacher tensors, import a model, schedule a GPU or train. A completed rerun verifies hashes and reads the saved summary without reselecting prompts or repeating overlap checks. This is a small implementation pilot; the general/hard balance is intentional for debugging and is not the later healing sampling policy.


In [1]:
# Freeze a small, split-safe teacher-target capture pilot; reuse saved selections on rerun.
import hashlib
import json
from pathlib import Path

root = Path('data/phase5')
root.mkdir(exist_ok=True)
prototype_manifest_path = Path('data/phase4/bridge_prototypes_request.json')
prototype_manifest = json.loads(prototype_manifest_path.read_text(encoding='utf-8'))
prototype = prototype_manifest['request']
prototype_summary_path = Path('data/phase4/bridge_prototypes_summary.json')
prototype_summary = json.loads(prototype_summary_path.read_text(encoding='utf-8'))
if prototype_summary['request_key'] != prototype_manifest['request_key']:
    raise RuntimeError('Prototype provenance differs.')
insertion_path = Path('data/phase4/insertion_block_05_smoke/summary.json')
insertion = json.loads(insertion_path.read_text(encoding='utf-8'))
if insertion['status'] != 'passed' or len(insertion['calls']) != 40:
    raise RuntimeError('Full-pipeline insertion smoke must pass first.')
contract_path = Path('data/phase4/bridge_block_contract_request.json')
contract = json.loads(contract_path.read_text(encoding='utf-8'))
insertion_request_path = Path('data/phase4/insertion_block_05_smoke/request.json')
insertion_request = json.loads(insertion_request_path.read_text(encoding='utf-8'))
target = insertion_request['request']['target']
protocol_path = Path('data/phase2/protocol.json')
protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
for field in ('model_repo', 'model_revision', 'config_sha256'):
    if prototype[field] != target[field]:
        raise RuntimeError('Phases 4 and 5 identify different teachers.')
if (protocol['MODEL_REPO'] != target['model_repo'] or protocol['MODEL_REVISION'] != target['model_revision']
        or protocol['NUM_INFERENCE_STEPS'] != 40 or target['dtype'] != 'bfloat16'):
    raise RuntimeError('Teacher protocol changed.')
source_paths = {
    'train': Path('data/phase1/pilot_20k/train.jsonl'),
    'validation': Path('data/phase1/pilot_20k/validation.jsonl'),
    'internal_test': Path('data/phase1/pilot_20k/internal_test.jsonl'),
    'qwen_image_bench': Path('data/qwen_image_bench/prompts.jsonl'),
    'phase3_probes': Path('data/phase3/layer_probe_set.jsonl'),
}
source_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in source_paths.items()}
for split in ('train', 'validation', 'internal_test'):
    if source_hashes[split] != prototype['split_sha256'][split]:
        raise RuntimeError('Phase 1 split changed after prototypes were frozen.')
artifact_paths = {
    'prototype_manifest': prototype_manifest_path, 'prototype_summary': prototype_summary_path,
    'insertion_summary': insertion_path, 'insertion_request': insertion_request_path,
    'contract_request': contract_path, 'phase2_protocol': protocol_path,
    'core': Path('data/phase4/bridge_definition.py'),
    'wrapper': Path('data/phase4/bridge_block_definition.py'),
}
artifact_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in artifact_paths.items()}
if (artifact_hashes['core'] != prototype['definition_sha256'] or
        artifact_hashes['wrapper'] != contract['request']['wrapper_sha256']):
    raise RuntimeError('Bridge source changed.')
for layer, digest in prototype_summary['checkpoint_sha256'].items():
    if hashlib.sha256(Path(f'data/phase4/bridge_block_{int(layer):02d}_initial.pt').read_bytes()).hexdigest() != digest:
        raise RuntimeError('Initial bridge checkpoint changed.')
dimensions = ['alignment', 'visual_quality', 'aesthetics', 'real_world', 'creative']
request = {
    'version': 'phase5-target-capture-pilot-manifest-v1',
    'model_repo': target['model_repo'], 'model_revision': target['model_revision'],
    'config_sha256': target['config_sha256'], 'num_teacher_blocks': prototype['num_teacher_blocks'],
    'hidden_dim': prototype['hidden_dim'], 'candidate_layers': prototype['candidate_layers'],
    'initial_checkpoint_sha256': prototype_summary['checkpoint_sha256'],
    'prototype_request_key': prototype_manifest['request_key'],
    'transformer_source_sha256': target['transformer_source_sha256'],
    'scheduler_class': target['scheduler_class'], 'scheduler_config': target['scheduler_config'],
    'runtime': target['runtime'], 'dtype': target['dtype'], 'noise_device': target['noise_device'],
    'num_inference_steps': 40, 'true_cfg_scale': protocol['TRUE_CFG_SCALE'],
    'resolutions': protocol['RESOLUTIONS'],
    'source_sha256': source_hashes, 'phase4_artifact_sha256': artifact_hashes,
    'dimensions': dimensions, 'selection_salt': 'nihonga-phase5-capture-pilot-v1',
    'selection': 'per dimension: TRAIN 2 general + 2 hard; VALIDATION 1 general + 1 hard; prefer distinct subdimensions using salted hash order',
    'stages': [{'name': 'early', 'step_index': 0}, {'name': 'middle', 'step_index': 20}, {'name': 'late', 'step_index': 39}],
    'capture': {
        'teacher_only': True, 'teacher_frozen': True, 'use_kv_cache': True,
        'image_token_cap_per_stage': 256, 'valid_prefix_token_cap_at_step_zero': 64,
        'token_selection': 'deterministic uniform without replacement within role; same indices across candidate layers; save actual indices and total available counts',
        'token_selection_seed_salt': 'nihonga-phase5-token-sampling-v1',
        'tensor_dtype': 'bfloat16', 'store': ['paired_block_input', 'paired_block_output', 'token_indices', 'token_roles', 'valid_token_mask'],
        'metadata': ['source_id', 'split', 'prompt', 'seed', 'width', 'height', 'step_index',
                     'scheduler_timestep', 'scheduler_sigma', 'cache_mode', 'full_hidden_shape',
                     'prefix_token_count', 'image_token_count', 'layer', 'request_key', 'tensor_sha256'],
        'prefix_policy': 'valid prefix only at extraction; cached stages capture target-image tokens only; padded text excluded',
        'teacher_inputs': 'save initial packed latent and prompt embeddings/masks once per prompt; save current packed latent and transformer-call metadata at each sampled stage',
        'persistence': 'commit each prompt packet remotely before return; verify and save locally; started-without-result or failures stop without automatic recapture',
    },
    'data_policy': {'train': 'gradient updates only', 'validation': 'selection and diagnostics only; never optimizer updates',
                    'excluded': ['internal_test', 'qwen_image_bench', 'phase3_probes'],
                    'duplicate_check': 'source IDs and NFKC/casefold/whitespace-normalized exact prompt matches'},
    'fitting_policy': 'independent single-block bridges; equal rank, same captured examples and optimizer budget for every candidate; no joint replacement',
    'scope': 'capture and fitting implementation pilot, not a representative evaluation or sufficient training dataset',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
manifest_path = root / 'capture_pilot_request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding='utf-8')) != manifest:
        raise RuntimeError('Saved capture request differs; preserve and inspect.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
rows_path = root / 'capture_pilot_rows.jsonl'
summary_path = root / 'capture_pilot_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    if summary['request_key'] != key or hashlib.sha256(rows_path.read_bytes()).hexdigest() != summary['rows_sha256']:
        raise RuntimeError('Saved capture pilot rows changed or are missing.')
    print('Reusing verified capture manifest and saved split checks; no sampling, model import or API call.')
else:
    from collections import Counter
    import unicodedata

    def normalize(text):
        return ' '.join(unicodedata.normalize('NFKC', text).casefold().split())

    datasets = {name: [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
                for name, path in source_paths.items()}
    def ids(rows):
        return {str(row.get('source_id', row.get('id', row.get('eval_id', row.get('ID', ''))))) for row in rows}
    def prompts(rows):
        return {normalize(row[field]) for row in rows for field in ('prompt', 'prompt_en', 'prompt_cn')
                if field in row and row[field]}
    forbidden_ids = set().union(*(ids(datasets[name]) for name in ('internal_test', 'qwen_image_bench', 'phase3_probes')))
    forbidden_prompts = set().union(*(prompts(datasets[name]) for name in ('internal_test', 'qwen_image_bench', 'phase3_probes')))
    selected = []
    for split, per_role in (('train', 2), ('validation', 1)):
        rows = datasets[split]
        if len(ids(rows)) != len(rows) or any(row['split'] != split for row in rows):
            raise RuntimeError('Source IDs are duplicated or split labels differ.')
        if set(row['dimension'] for row in rows) != set(dimensions):
            raise RuntimeError('Capability groups differ from the capture policy.')
        for dimension in dimensions:
            for hard in (False, True):
                pool = [row for row in rows if row['dimension'] == dimension and row['hard'] == hard
                        and row['id'] not in forbidden_ids and normalize(row['prompt']) not in forbidden_prompts]
                def rank(row):
                    return hashlib.sha256(f"{request['selection_salt']}:{split}:{row['id']}".encode()).hexdigest()
                pool.sort(key=rank)
                chosen, used_subdimensions = [], set()
                for row in pool:
                    if row['subdimension'] not in used_subdimensions:
                        chosen.append(row)
                        used_subdimensions.add(row['subdimension'])
                    if len(chosen) == per_role:
                        break
                if len(chosen) < per_role:
                    chosen.extend(row for row in pool if row not in chosen)
                    chosen = chosen[:per_role]
                if len(chosen) != per_role:
                    raise RuntimeError('Insufficient eligible rows for capture pilot.')
                for source in chosen:
                    width, height = request['resolutions'][source['aspect_ratio']]
                    record = dict(source)
                    record.update(capture_id=f"p5-{split}-{source['id']}", source_id=source['id'],
                                  width=width, height=height, num_inference_steps=40,
                                  true_cfg_scale=request['true_cfg_scale'],
                                  usage='gradient_updates' if split == 'train' else 'validation_only')
                    selected.append(record)
    train = [row for row in selected if row['split'] == 'train']
    validation = [row for row in selected if row['split'] == 'validation']
    id_overlap = ids(train) & ids(validation)
    prompt_overlap = prompts(train) & prompts(validation)
    forbidden_id_overlap = ids(selected) & forbidden_ids
    forbidden_prompt_overlap = prompts(selected) & forbidden_prompts
    if id_overlap or prompt_overlap or forbidden_id_overlap or forbidden_prompt_overlap or len(prompts(selected)) != len(selected):
        raise RuntimeError('Capture selection has duplicate prompts or evaluation overlap.')
    raw = ''.join(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n' for row in selected).encode()
    if rows_path.exists():
        if rows_path.read_bytes() != raw:
            raise RuntimeError('Partial saved selection differs; do not overwrite.')
    else:
        with rows_path.open('xb') as handle:
            handle.write(raw)
    n_prompts = len(selected)
    n_layers = len(request['candidate_layers'])
    # Pair payload upper bound: input + output, each BF16, excluding replay inputs and metadata.
    max_tokens_per_prompt_layer = 3 * 256 + 64
    max_pair_bytes = n_prompts * n_layers * max_tokens_per_prompt_layer * request['hidden_dim'] * 2 * 2
    summary = {
        'request_key': key, 'status': 'manifest_ready; no teacher targets captured',
        'rows_sha256': hashlib.sha256(raw).hexdigest(), 'n_prompts': n_prompts,
        'split_counts': dict(Counter(row['split'] for row in selected)),
        'dimension_counts_by_split': {split: dict(Counter(row['dimension'] for row in selected if row['split'] == split)) for split in ('train', 'validation')},
        'hard_counts_by_split': {split: dict(Counter('hard' if row['hard'] else 'general' for row in selected if row['split'] == split)) for split in ('train', 'validation')},
        'aspect_ratio_counts': dict(Counter(row['aspect_ratio'] for row in selected)),
        'difficulty_counts': dict(Counter(str(row['difficulty']) for row in selected)),
        'subdimension_count': len(set((row['dimension'], row['subdimension']) for row in selected)),
        'planned_teacher_trajectories': n_prompts, 'planned_teacher_transformer_calls': n_prompts * 40,
        'planned_stage_snapshots': n_prompts * 3, 'planned_block_pair_records': n_prompts * 3 * n_layers,
        'maximum_selected_token_pairs': n_prompts * n_layers * max_tokens_per_prompt_layer,
        'maximum_pair_tensor_bytes': max_pair_bytes, 'captured_block_pair_records': 0, 'optimizer_updates': 0,
        'leakage_checks': {'train_validation_id_overlap': len(id_overlap), 'train_validation_normalized_prompt_overlap': len(prompt_overlap),
                           'evaluation_id_overlap': len(forbidden_id_overlap), 'evaluation_normalized_prompt_overlap': len(forbidden_prompt_overlap)},
        'interpretation': 'small balanced implementation pilot; upper bound covers paired BF16 hidden tensors only; no quality or fit evidence',
    }
    with summary_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(summary, indent=2) + '\n')
print('Capture manifest:', summary['status'])
print('Prompts:', summary['split_counts'], '| candidates:', request['candidate_layers'])
print('Planned block-pair records:', summary['planned_block_pair_records'], '| teacher trajectories:', summary['planned_teacher_trajectories'])
print('Paired-hidden-tensor upper bound GiB:', summary['maximum_pair_tensor_bytes'] / 2**30)
print('Exact-match overlap checks:', summary['leakage_checks'])
print('Captured records:', summary['captured_block_pair_records'], '| optimizer updates:', summary['optimizer_updates'])
print('Artifacts:', root)


Reusing verified capture manifest and saved split checks; no sampling, model import or API call.
Capture manifest: manifest_ready; no teacher targets captured
Prompts: {'train': 20, 'validation': 10} | candidates: [2, 4, 5, 3]
Planned block-pair records: 360 | teacher trajectories: 30
Paired-hidden-tensor upper bound GiB: 1.5234375
Exact-match overlap checks: {'train_validation_id_overlap': 0, 'train_validation_normalized_prompt_overlap': 0, 'evaluation_id_overlap': 0, 'evaluation_normalized_prompt_overlap': 0}
Captured records: 0 | optimizer updates: 0
Artifacts: data\phase5


## What the capture-manifest results mean

**The capture pilot is ready; no teacher targets or learned weights have been produced.** We saved 30 exact prompt/seed/resolution records and a contract describing what teacher capture must persist. The same teacher examples will supervise all four independent bridge candidates, allowing a fair fitting comparison without repeating a teacher trajectory for each candidate.

| Capability group | TRAIN prompts | VALIDATION prompts |
|---|---:|---:|
| Prompt alignment | 4 | 2 |
| Visual quality | 4 | 2 |
| Aesthetics | 4 | 2 |
| Real-world fidelity | 4 | 2 |
| Creative generation | 4 | 2 |
| Total | 20 | 10 |

TRAIN contains **10 general and 10 hard prompts**; VALIDATION contains **5 general and 5 hard prompts**. Across both splits, the selection spans **21 subdimensions**, all five difficulty labels, and **7 aspect ratios**. Difficulty counts are 5 at level 1, 2 at level 2, 11 at level 3, 9 at level 4 and 3 at level 5. Aspect-ratio counts are 12 square, 7 at 16:9, 3 at 2:3, 2 at 3:2, 3 at 3:4, 1 at 9:16 and 2 at 4:3. The pilot does not cover 21:9. These are coverage counts, not quality measurements; 30 prompts cannot establish generalization across the full dataset.

All four overlap counts are **zero**: TRAIN versus VALIDATION source IDs, TRAIN versus VALIDATION normalized prompt text, selected IDs versus evaluation IDs, and selected normalized prompt text versus evaluation text. Benchmark checks include English and Chinese prompts. This supports the chosen split boundaries for this pilot. Exact normalized matching does not detect all paraphrases, semantic similarity, or related template families; these checks are not proof of complete semantic separation.

| Planned quantity | Count | Meaning |
|---|---:|---|
| Teacher trajectories | 30 | One 40-step teacher run per selected prompt |
| Teacher transformer calls | 1,200 | 30 trajectories × 40 denoising steps at guidance scale 1 |
| Stage snapshots | 90 | 30 prompts × 3 sampled denoising stages |
| Block input/output records | 360 | 90 snapshots × 4 candidate blocks |
| Maximum selected token pairs | 99,840 | Each pair has a teacher input and target output |
| Captured block records | 0 | Capture has not run |
| Optimizer updates | 0 | Fitting has not run |

The 360 records divide into **240 TRAIN records and 120 VALIDATION records**. They are expected records, not existing tensor files. Early/middle/late name the positions in the 40-step trajectory; the actual scheduler timesteps and noise levels must be measured during capture, rather than assumed from those labels.

The selected BF16 input/output hidden tensors have an upper bound of **1,635,778,560 bytes (1.52 GiB)**. This counts 30 prompts × 4 layers × (320 early-stage tokens + 256 middle-stage tokens + 256 late-stage tokens) × 4096 components × 2 tensors × 2 bytes. Early-stage prefix counts may be smaller than 64. The estimate excludes embeddings, replay latents, token metadata, serialization overhead and remote backups, and is not a GPU-memory or training-memory estimate. The cap controls capture size; it does not shorten the teacher's sequence or reduce the planned 40-step trajectories.

**Decision supported:** capture this small split-safe pilot before committing to a larger teacher-target dataset. **Still uncertain:** whether the captured tensors and replay inputs are sufficient and correctly aligned, whether prefix/image sampling and three stages reveal the bridge's main limitations, and whether this adapter can learn a useful approximation. Independent fitting, held-out validation and full-model evaluation remain necessary. The pilot is far smaller than the 10k-prompt smoke-training scale suggested in PLAN.md; it tests implementation, not convergence or production quality.

Saved artifacts: `data/phase5/capture_pilot_request.json`, `capture_pilot_rows.jsonl` and `capture_pilot_summary.json`. Initial phase 4 checkpoints are unchanged. A completed rerun of all phase 4 code cells and this phase 5 cell changed no result artifact, imported no model or network client, and performed no prompt selection or overlap calculation.


## 2. Capture and verify one TRAIN teacher trajectory

We capture the first saved TRAIN prompt on the pinned Modal H100 runtime. All teacher components remain frozen; the four candidate blocks stay intact. Forward hooks save sampled paired block inputs and outputs at steps 0, 20 and 39, with the same token indices shared across candidate layers. Prefix text is sampled only during extraction, and padded text is excluded. The full 40-step trajectory produces latents without decoding an image.

We save the initial packed latent, prompt embeddings and masks, stage latents and exact transformer call metadata, sampled token indices and roles, teacher velocities, and the terminal latent. Three saved teacher inputs are replayed once with a fresh teacher prefix cache. Their velocity predictions must match the captured teacher outputs within relative L2 tolerance 0.001; all control outputs are persisted. These controls check capture alignment, not student quality.

Identity input/output errors are reported separately for image tokens at each stage and prefix text at extraction. They describe the missing teacher block update before any bridge learning. The tensor bundle and report are committed remotely before download, checksummed locally, and inspected once through weights-only loading. A completed rerun reads saved reports and hashes without model imports or calls. Failed or incomplete remote captures stop instead of automatically repeating expensive work.


In [2]:
# Capture one TRAIN trajectory with paired block states; completed reruns only verify saved local files.
import hashlib
import json
from pathlib import Path

root = Path('data/phase5')
manifest_path = root / 'capture_pilot_request.json'
capture_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
config = capture_manifest['request']
if hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest() != capture_manifest['request_key']:
    raise RuntimeError('Capture contract checksum differs.')
pilot_summary = json.loads((root / 'capture_pilot_summary.json').read_text(encoding='utf-8'))
row_bytes = (root / 'capture_pilot_rows.jsonl').read_bytes()
if (pilot_summary['request_key'] != capture_manifest['request_key'] or
        hashlib.sha256(row_bytes).hexdigest() != pilot_summary['rows_sha256']):
    raise RuntimeError('Capture pilot selections differ.')
rows = [json.loads(line) for line in row_bytes.decode().splitlines() if line.strip()]
row = next(row for row in rows if row['split'] == 'train')
if row['usage'] != 'gradient_updates' or row['true_cfg_scale'] != 1.0:
    raise RuntimeError('Smoke capture must use a TRAIN prompt at guidance scale 1.')
worker_path = root / 'teacher_capture_worker.py'
worker_source = worker_path.read_text(encoding='utf-8')
request = {'version': 'phase5-one-prompt-teacher-capture-v1',
           'capture_manifest_key': capture_manifest['request_key'], 'capture_request': config,
           'rows_sha256': pilot_summary['rows_sha256'], 'row': row,
           'worker_sha256': hashlib.sha256(worker_source.encode()).hexdigest(),
           'replay_relative_l2_tolerance': 1e-3,
           'scope': 'first TRAIN prompt only; 40 teacher steps plus 3 persisted replay controls; no training'}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = root / 'teacher_targets' / row['capture_id']
run_dir.mkdir(parents=True, exist_ok=True)
request_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if request_path.exists():
    if json.loads(request_path.read_text(encoding='utf-8')) != manifest:
        raise RuntimeError('Capture implementation or inputs changed; preserve the original capture.')
else:
    with request_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2, ensure_ascii=False) + '\n')
report_path = run_dir / 'summary.json'
tensor_path = run_dir / 'targets.pt'
remote_directory = f'/phase5/capture/{key}'

def hash_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def validate(report):
    if report['request_key'] != key or report['row'] != row or report['capture_manifest_key'] != capture_manifest['request_key']:
        raise RuntimeError('Capture report provenance differs.')
    if report['status'] != 'passed':
        raise RuntimeError('Saved capture failed; no automatic retry: ' + report.get('error', 'unknown'))
    if (report['captured_block_pair_records'] != 12 or report['teacher_trajectory_calls'] != 40
            or report['teacher_replay_calls'] != 3 or report['optimizer_updates'] != 0):
        raise RuntimeError('Capture report has unexpected counts.')

report = None
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    validate(report)
    if tensor_path.exists():
        if hash_file(tensor_path) != report['tensor_sha256']:
            raise RuntimeError('Saved tensor bundle checksum differs.')
        print('Reusing verified local teacher targets and controls; no Modal call, model import or recapture.')
    else:
        report = None
if report is None:
    import modal
    runtime = config['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        report = json.loads(b''.join(results_volume.read_file(remote_directory + '/summary.json')))
        print('Recovering saved remote capture; no GPU scheduled.')
    except FileNotFoundError:
        if report_path.exists() or tensor_path.exists():
            raise RuntimeError('Incomplete local capture has no remote report; never recapture automatically.')
        app = modal.App('nihonga-phase5-teacher-capture-smoke')
        weights = modal.Volume.from_name(runtime['weights_volume'])
        image = (modal.Image.debian_slim(python_version=runtime['python_version'])
                 .apt_install('git').pip_install(*runtime['packages'])
                 .env({'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'}))
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu=runtime['gpu_requested'], serialized=True,
            timeout=1800, max_containers=1, retries=0,
            volumes={'/root/.cache/huggingface': weights.with_mount_options(read_only=True),
                     '/phase3': results_volume})(namespace['capture_one'])
        with modal.enable_output(), app.run():
            report = worker.remote(request, key)
    if report['request_key'] != key:
        raise RuntimeError('Remote capture report belongs to different inputs.')
    if report['status'] == 'passed':
        if not tensor_path.exists():
            temporary = tensor_path.with_suffix('.download')
            with temporary.open('wb') as handle:
                for chunk in results_volume.read_file(remote_directory + '/targets.pt'):
                    handle.write(chunk)
            if hash_file(temporary) != report['tensor_sha256']:
                raise RuntimeError('Downloaded teacher targets checksum differs.')
            temporary.replace(tensor_path)
        elif hash_file(tensor_path) != report['tensor_sha256']:
            raise RuntimeError('Local targets differ from the remote capture.')
    if report_path.exists():
        if json.loads(report_path.read_text(encoding='utf-8')) != report:
            raise RuntimeError('Local capture report differs from backup.')
    else:
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
    validate(report)

inspection_path = run_dir / 'local_tensor_inspection.json'
if inspection_path.exists():
    inspection = json.loads(inspection_path.read_text(encoding='utf-8'))
    if inspection['request_key'] != key or inspection['tensor_sha256'] != report['tensor_sha256'] or inspection['status'] != 'passed':
        raise RuntimeError('Saved local tensor inspection differs or failed.')
else:
    import torch
    bundle = torch.load(tensor_path, map_location='cpu', weights_only=True)
    if bundle['request_key'] != key or bundle['row'] != row or len(bundle['pairs']) != 12 or len(bundle['stages']) != 3:
        raise RuntimeError('Downloaded tensor bundle has unexpected provenance or counts.')
    pairs_by_stage = {}
    for pair in bundle['pairs']:
        if pair['input'].dtype != torch.bfloat16 or pair['input'].shape != pair['target'].shape or pair['input'].shape[-1] != 4096:
            raise RuntimeError('Downloaded paired tensors have unexpected shape/dtype.')
        if not torch.isfinite(pair['input']).all() or not torch.isfinite(pair['target']).all():
            raise RuntimeError('Downloaded paired tensors are nonfinite.')
        if not pair['valid_token_mask'].all() or pair['token_indices'].numel() != pair['input'].shape[1]:
            raise RuntimeError('Saved sampled-token metadata is inconsistent.')
        if pair['stage'] in pairs_by_stage:
            reference = pairs_by_stage[pair['stage']]
            if not torch.equal(pair['token_indices'], reference['token_indices']) or not torch.equal(pair['token_roles'], reference['token_roles']):
                raise RuntimeError('Candidates use different sampled tokens.')
        else:
            pairs_by_stage[pair['stage']] = pair
    inspection = {'request_key': key, 'tensor_sha256': report['tensor_sha256'], 'status': 'passed',
                  'weights_only_load': True, 'paired_shapes_and_finiteness': True, 'shared_token_indices': True,
                  'captured_block_pair_records': len(bundle['pairs'])}
    with inspection_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(inspection, indent=2) + '\n')
    del bundle
print('Teacher capture:', report['status'], '| source:', row['source_id'], '| split:', row['split'])
print('Paired block records:', report['captured_block_pair_records'], '| sampled token pairs:', report['selected_token_pairs'])
print('Bundle MiB:', report['tensor_bytes'] / 2**20, '| local inspection:', inspection['status'])
for stage in report['stages']:
    print(stage['stage'], '| step:', stage['step_index'], '| sigma:', stage['scheduler_sigma'], '| transformer timestep:', stage['transformer_timestep'])
print('Replay control relative L2 errors:', [item['relative_l2_error'] for item in report['replay_controls']])
for item in report['identity_metrics']:
    print(f"Block {item['layer']} | {item['stage']} | {item['role']} | identity relative L2: {item['identity_relative_l2_error']:.6f} | mean cosine: {item['identity_mean_token_cosine']:.6f}")
print('Optimizer updates:', report['optimizer_updates'], '| saved targets:', tensor_path)


Reusing verified local teacher targets and controls; no Modal call, model import or recapture.
Teacher capture: passed | source: p1-2eea6009c17f | split: train
Paired block records: 12 | sampled token pairs: 3232
Bundle MiB: 55.84724140167236 | local inspection: passed
early | step: 0 | sigma: 1.0 | transformer timestep: [1.0]
middle | step: 20 | sigma: 0.6566662788391113 | transformer timestep: [0.65625]
late | step: 39 | sigma: 0.019999980926513672 | transformer timestep: [0.02001953125]
Replay control relative L2 errors: [0.0, 0.0, 0.0]
Block 2 | early | prefix_text | identity relative L2: 0.074209 | mean cosine: 0.990623
Block 2 | early | target_image | identity relative L2: 0.080438 | mean cosine: 0.998396
Block 3 | early | prefix_text | identity relative L2: 0.069918 | mean cosine: 0.990949
Block 3 | early | target_image | identity relative L2: 0.105680 | mean cosine: 0.997141
Block 4 | early | prefix_text | identity relative L2: 0.120180 | mean cosine: 0.990427
Block 4 | early |

## What the first teacher capture results mean

**We now have actual teacher-supervised training pairs for one TRAIN prompt.** The intact, frozen 32-block teacher processed `p1-2eea6009c17f`, a Chinese alignment/materials prompt describing a knitted wool vase beside a carbon-fiber fish sculpture. Its saved seed is **787111945**, at **1024 × 1024** pixels. The prompt is from the frozen TRAIN selection and is eligible for bridge fitting; no validation or evaluation example was used for this capture.

The run produced **12 block input/output records**: four candidates (2, 3, 4 and 5) at three stages. Each record contains sampled BF16 inputs and the corresponding teacher block outputs with width **4096**. Each stage uses **256 sampled image tokens per layer**; extraction also includes **40 valid prefix-text tokens per layer**. Later cached stages contain image queries only. This gives **3,232 sampled token pairs**, including both token roles. The candidates share exactly the same token indices at each stage. The local weights-only load checked pair shapes, finite values, masks and shared sampling indices.

| Stage | Zero-based denoising step | Scheduler timestep | Sigma | Actual BF16 transformer input timestep | Cache mode |
|---|---:|---:|---:|---:|---|
| early | 0 | 1000.000000 | 1.000000 | 1.00000000 | extract |
| middle | 20 | 656.666260 | 0.656666 | 0.65625000 | cached |
| late | 39 | 19.999981 | 0.020000 | 0.02001953 | cached |

Sigma describes the current noise level used by the scheduler. The pipeline divides its scheduler timestep by 1000 in BF16, so the actual transformer input is rounded: the middle input is **0.65625** and the late input is **0.02001953**. Both the scheduler values and actual input tensors are saved for exact replay. The middle-stage sigma is measured from the shifted schedule; it is not assumed to equal 0.5. These values make the cached examples traceable to their denoising stage.

**Replay controls passed:** the same intact teacher replayed the three saved transformer inputs with a fresh prefix cache built from the early input.

| Stage | Relative L2 velocity difference | Maximum absolute velocity difference |
|---|---:|---:|
| early | 0 | 0 |
| middle | 0 | 0 |
| late | 0 | 0 |

Relative L2 is the norm of the prediction difference divided by the norm of the captured teacher velocity. The saved pass tolerance is **0.001 (0.1%)**. These controls support the alignment of latents, prompt conditioning, timestep and cache handling. They compare teacher with teacher, not bridge with teacher, and establish no image-quality result. The three replay outputs are saved, so this verification does not repeat on rerun.

### Teacher block updates that an identity bridge misses

The following measurements compare each sampled block input with its teacher output. Relative L2 error measures the omitted block update as a fraction of the target output norm. Token cosine measures average directional agreement, with 1 meaning the same direction. High cosine can coexist with a substantial missing update; it is not enough to justify removal. These are local hidden-state measurements on one training trajectory, not output-velocity errors or image-quality scores.

| Block | Stage | Identity relative L2 error, image tokens | Mean token cosine |
|---|---|---:|---:|
| 2 | early | 8.04% | 0.998396 |
| 3 | early | 10.57% | 0.997141 |
| 4 | early | 8.09% | 0.997225 |
| 5 | early | 10.17% | 0.995872 |
| 2 | middle | 5.41% | 0.998621 |
| 3 | middle | 7.59% | 0.998213 |
| 4 | middle | 9.66% | 0.997539 |
| 5 | middle | 11.14% | 0.996473 |
| 2 | late | 8.39% | 0.997683 |
| 3 | late | 8.85% | 0.997886 |
| 4 | late | 10.91% | 0.997399 |
| 5 | late | 9.63% | 0.997022 |

Prefix text is reported separately because it uses the zero-timestep conditioning path and influences downstream image attention through cached keys and values.

| Block | Sampled prefix tokens | Identity relative L2 error, prefix text | Mean token cosine |
|---|---:|---:|---:|
| 2 | 40 | 7.42% | 0.990623 |
| 3 | 40 | 6.99% | 0.990949 |
| 4 | 40 | 12.02% | 0.990427 |
| 5 | 40 | 49.10% | 0.982414 |

For this prompt, image-token identity errors range from **5.41% to 11.14%** of the teacher output norm. Block 5's prefix-text error is **49.10%**, even though its mean token cosine is **0.982414**. This shows why directional agreement alone can hide a large update in magnitude. It supports monitoring raw relative errors separately from normalized fitting loss, and reporting prefix text separately from image tokens. It does not establish that block 5 is unsuitable for a learned bridge.

The report also saves normalized MSE and teacher target RMS for each stage/role. Normalized MSE averages squared differences after per-token L2 normalization across all 4096 components; its small scale must be interpreted with that averaging in mind. Target RMS describes the target's magnitude. Together with relative L2 and cosine, these diagnostics distinguish direction from magnitude errors. No learned bridge has been evaluated yet, and this single prompt cannot rank the four candidates reliably.

The saved bundle occupies **55.85 MiB**. It contains the paired states, sampled indices and role/mask metadata, shared prompt embeddings and masks, initial packed latent, the three stage latents and exact call metadata, captured teacher velocities, replay velocities and terminal latent. No image was decoded. Each paired record links to its stage's measured timestep, sigma and teacher input. The bundle's SHA-256 matches the remote report and the local inspection. Source prompt, seed, resolution, split and request keys are included for provenance.

The 40-step capture took **9.61 seconds**, excluding model loading and the three replay controls. Peak allocated GPU memory during capture was **30.86 GiB**. Hooks copy selected tensors to CPU and synchronize checks; this is a capture-readiness measurement, not a latency benchmark or a training-memory estimate. Exactly **40 trajectory calls and 3 replay calls** ran in this first capture.

**Decision supported:** the capture implementation produces finite, aligned, reloadable teacher targets for the tested prompt. **Still uncertain:** behavior across the other prompt groups, resolutions and validation examples; whether sampled tokens represent the full hidden sequences well; and whether training the cheap adapters recovers the missing updates. We have **12 of the planned 360 records** (one of 30 trajectories), all from TRAIN, and **zero optimizer updates**. No phase 4 checkpoint changed. Full-model condition-image capture and larger training datasets remain untested.

Files are saved under `data/phase5/teacher_targets/p5-train-p1-2eea6009c17f/`: `targets.pt`, `summary.json`, `request.json`, and `local_tensor_inspection.json`. The teacher tensor bundle, request and report are also committed to the existing remote results volume. The capture implementation is saved in `data/phase5/teacher_capture_worker.py`. A completed rerun of all phase 4 and phase 5 code cells changed no saved artifact and imported no Torch, Diffusers, Modal or network client; it performed no capture, replay, tensor inspection, selection or optimization.


## 3. Complete the frozen TRAIN and VALIDATION capture pilot

We expand the verified paired-state capture to all 30 saved prompts: 20 TRAIN and 10 VALIDATION. The first saved TRAIN capture is reused unchanged. The batch worker loads the pinned teacher once and preserves the tested capture logic while processing the remaining prompts independently with their own saved seeds and fresh prefix caches. All teacher components stay frozen; no bridge is trained.

Each prompt yields 12 paired records and three teacher replay controls under the existing capture contract. The worker commits each prompt remotely before returning its report; the notebook downloads and checksums each completed tensor bundle and records a one-time local weights-only inspection. Existing local or remote results are reused. A started prompt without a completed report, failed capture, or checksum mismatch stops the batch without automatic recapture.

We save separate TRAIN and VALIDATION entries in a target index and aggregate identity errors by split, candidate block, denoising stage and token role. These are unweighted means across prompts; prefix text remains separate from image tokens. Completed reruns verify saved files and summaries without importing models, replaying inputs or recalculating metrics. This step expands the implementation pilot, not the full training dataset.


In [3]:
# Complete the frozen 30-prompt capture pilot, preserving every existing prompt result.
import hashlib
import json
from pathlib import Path

root = Path('data/phase5')
capture_manifest = json.loads((root / 'capture_pilot_request.json').read_text(encoding='utf-8'))
config = capture_manifest['request']
if hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest() != capture_manifest['request_key']:
    raise RuntimeError('Capture manifest checksum differs.')
pilot = json.loads((root / 'capture_pilot_summary.json').read_text(encoding='utf-8'))
row_bytes = (root / 'capture_pilot_rows.jsonl').read_bytes()
if hashlib.sha256(row_bytes).hexdigest() != pilot['rows_sha256'] or pilot['request_key'] != capture_manifest['request_key']:
    raise RuntimeError('Frozen pilot rows differ.')
rows = [json.loads(line) for line in row_bytes.decode('utf-8').splitlines() if line.strip()]
if len(rows) != 30 or sum(row['split'] == 'train' for row in rows) != 20 or sum(row['split'] == 'validation' for row in rows) != 10:
    raise RuntimeError('Capture pilot split counts differ.')
worker_path = root / 'teacher_capture_batch_worker.py'
worker_source = worker_path.read_text(encoding='utf-8')
worker_hash = hashlib.sha256(worker_source.encode()).hexdigest()
first_path = root / 'teacher_targets' / rows[0]['capture_id'] / 'request.json'
first_manifest = json.loads(first_path.read_text(encoding='utf-8'))
if (first_manifest['request']['row'] != rows[0] or first_manifest['request']['capture_request'] != config
        or first_manifest['request']['rows_sha256'] != pilot['rows_sha256']
        or first_manifest['request']['worker_sha256'] != hashlib.sha256((root / 'teacher_capture_worker.py').read_bytes()).hexdigest()):
    raise RuntimeError('Verified smoke capture differs from the frozen pilot.')
tasks = []
for i, row in enumerate(rows):
    if i == 0:
        manifest = first_manifest
    else:
        request = {'version': 'phase5-pilot-prompt-teacher-capture-v2',
                   'capture_manifest_key': capture_manifest['request_key'], 'capture_request': config,
                   'rows_sha256': pilot['rows_sha256'], 'row': row, 'worker_sha256': worker_hash,
                   'replay_relative_l2_tolerance': 1e-3,
                   'scope': 'full pilot prompt; 40 teacher steps plus 3 persisted replay controls; no training'}
        key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
        manifest = {'request': request, 'request_key': key}
    if hashlib.sha256(json.dumps(manifest['request'], sort_keys=True).encode()).hexdigest() != manifest['request_key']:
        raise RuntimeError('Per-prompt request key differs.')
    directory = root / 'teacher_targets' / row['capture_id']
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / 'request.json'
    if path.exists():
        if json.loads(path.read_text(encoding='utf-8')) != manifest:
            raise RuntimeError('A saved prompt request changed; preserve its targets.')
    else:
        with path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(manifest, indent=2, ensure_ascii=False) + '\n')
    tasks.append(manifest)
batch_request = {'version': 'phase5-full-capture-pilot-v1', 'capture_request': config,
                 'capture_manifest_key': capture_manifest['request_key'], 'rows_sha256': pilot['rows_sha256'],
                 'worker_sha256': worker_hash, 'prompt_request_keys': [task['request_key'] for task in tasks],
                 'existing_smoke_request_key': first_manifest['request_key'],
                 'policy': 'one teacher load; per-prompt durable commits; recover remote reports before scheduling GPU; no automatic recapture'}
batch_key = hashlib.sha256(json.dumps(batch_request, sort_keys=True).encode()).hexdigest()
batch_path = root / 'capture_full_request.json'
batch_manifest = {'request': batch_request, 'request_key': batch_key}
if batch_path.exists():
    if json.loads(batch_path.read_text(encoding='utf-8')) != batch_manifest:
        raise RuntimeError('Batch implementation or inputs changed.')
else:
    with batch_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(batch_manifest, indent=2) + '\n')

def hash_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def directory_for(task):
    return root / 'teacher_targets' / task['request']['row']['capture_id']

def validate(report, task):
    if (report['request_key'] != task['request_key'] or report['row'] != task['request']['row']
            or report['capture_manifest_key'] != capture_manifest['request_key']):
        raise RuntimeError('Per-prompt capture provenance differs.')
    if report['status'] != 'passed':
        raise RuntimeError('Saved capture failed; no automatic retry: ' + report.get('error', 'unknown'))
    if (report['captured_block_pair_records'] != 12 or report['teacher_trajectory_calls'] != 40
            or report['teacher_replay_calls'] != 3 or report['optimizer_updates'] != 0 or not report['teacher_frozen']):
        raise RuntimeError('Unexpected per-prompt capture counts or training state.')
    if any(item['relative_l2_error'] > task['request']['replay_relative_l2_tolerance'] for item in report['replay_controls']):
        raise RuntimeError('Saved replay control exceeds tolerance.')

reports, missing = {}, []
for task in tasks:
    directory = directory_for(task)
    report_path = directory / 'summary.json'
    tensor_path = directory / 'targets.pt'
    if report_path.exists():
        report = json.loads(report_path.read_text(encoding='utf-8'))
        validate(report, task)
        if tensor_path.exists():
            if hash_file(tensor_path) != report['tensor_sha256']:
                raise RuntimeError('Saved teacher tensor bundle changed.')
            reports[task['request_key']] = report
            continue
    missing.append(task)

if missing:
    import modal
    runtime = config['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_names = [entry.path.replace('\\', '/') for entry in results_volume.listdir('/phase5/capture', recursive=True)]
    except (FileNotFoundError, modal.exception.NotFoundError):
        remote_names = []
    def exists_remote(key, name):
        return any(path.endswith(f'/{key}/{name}') for path in remote_names)

    def receive(report, task):
        if report['request_key'] != task['request_key'] or report['row'] != task['request']['row']:
            raise RuntimeError('Remote report provenance differs.')
        directory = directory_for(task)
        tensor_path = directory / 'targets.pt'
        report_path = directory / 'summary.json'
        if report['status'] == 'passed':
            if not tensor_path.exists():
                temporary = tensor_path.with_suffix('.download')
                with temporary.open('wb') as handle:
                    for chunk in results_volume.read_file(f"/phase5/capture/{task['request_key']}/targets.pt"):
                        handle.write(chunk)
                if hash_file(temporary) != report['tensor_sha256']:
                    raise RuntimeError('Downloaded bundle checksum differs.')
                temporary.replace(tensor_path)
            elif hash_file(tensor_path) != report['tensor_sha256']:
                raise RuntimeError('Local tensor bundle differs from remote backup.')
        if report_path.exists():
            if json.loads(report_path.read_text(encoding='utf-8')) != report:
                raise RuntimeError('Local report differs from remote backup.')
        else:
            with report_path.open('x', encoding='utf-8') as handle:
                handle.write(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
        validate(report, task)
        reports[task['request_key']] = report
        print(f"Saved {len(reports)}/30 prompt captures: {report['row']['split']} / {report['row']['dimension']}.", flush=True)

    pending = []
    for task in missing:
        key = task['request_key']
        directory = directory_for(task)
        if exists_remote(key, 'summary.json'):
            report = json.loads(b''.join(results_volume.read_file(f'/phase5/capture/{key}/summary.json')))
            receive(report, task)
        else:
            if ((directory / 'summary.json').exists() or (directory / 'targets.pt').exists()
                    or exists_remote(key, 'started.json') or exists_remote(key, 'targets.pt')):
                raise RuntimeError('Partial prompt capture has no report backup; inspect without recapturing.')
            pending.append(task)
    if pending:
        app = modal.App('nihonga-phase5-teacher-capture-pilot')
        weights = modal.Volume.from_name(runtime['weights_volume'])
        image = (modal.Image.debian_slim(python_version=runtime['python_version'])
                 .apt_install('git').pip_install(*runtime['packages'])
                 .env({'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'}))
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu=runtime['gpu_requested'], serialized=True,
            timeout=3600, max_containers=1,
            volumes={'/root/.cache/huggingface': weights.with_mount_options(read_only=True),
                     '/phase3': results_volume})(namespace['capture_many'])
        with modal.enable_output(), app.run():
            by_key = {task['request_key']: task for task in tasks}
            for report in worker.remote_gen(batch_request, batch_key, pending):
                receive(report, by_key[report['request_key']])
else:
    print('Reusing all 30 verified local captures; no Modal calls or teacher forwards.')
if len(reports) != 30:
    raise RuntimeError('Capture pilot incomplete; completed prompt artifacts remain saved.')

# Inspect each newly downloaded bundle once; existing inspection records are reused.
for task in tasks:
    directory = directory_for(task)
    report = reports[task['request_key']]
    path = directory / 'local_tensor_inspection.json'
    if path.exists():
        inspection = json.loads(path.read_text())
        if (inspection['request_key'] != task['request_key'] or inspection['tensor_sha256'] != report['tensor_sha256']
                or inspection['status'] != 'passed'):
            raise RuntimeError('Saved local inspection differs or failed.')
        continue
    import torch
    bundle = torch.load(directory / 'targets.pt', map_location='cpu', weights_only=True)
    if bundle['request_key'] != task['request_key'] or bundle['row'] != task['request']['row'] or len(bundle['pairs']) != 12:
        raise RuntimeError('Downloaded bundle provenance differs.')
    seen, layouts = set(), {}
    for pair in bundle['pairs']:
        if (pair['input'].dtype != torch.bfloat16 or pair['target'].dtype != torch.bfloat16
                or pair['input'].shape != pair['target'].shape or pair['input'].shape[-1] != 4096
                or not torch.isfinite(pair['input']).all() or not torch.isfinite(pair['target']).all()
                or not pair['valid_token_mask'].all() or pair['token_indices'].numel() != pair['input'].shape[1]):
            raise RuntimeError('Downloaded paired tensors have invalid shape, values or metadata.')
        seen.add((pair['stage'], pair['layer']))
        if pair['stage'] in layouts:
            previous = layouts[pair['stage']]
            if not torch.equal(previous['token_indices'], pair['token_indices']) or not torch.equal(previous['token_roles'], pair['token_roles']):
                raise RuntimeError('Candidate sampling indices differ.')
        layouts[pair['stage']] = pair
    if seen != {(stage['name'], layer) for stage in config['stages'] for layer in config['candidate_layers']}:
        raise RuntimeError('Bundle is missing a stage or candidate.')
    inspection = {'request_key': task['request_key'], 'tensor_sha256': report['tensor_sha256'], 'status': 'passed',
                  'weights_only_load': True, 'paired_shapes_and_finiteness': True, 'shared_token_indices': True,
                  'captured_block_pair_records': len(bundle['pairs'])}
    with path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(inspection, indent=2) + '\n')
    del bundle

summary_path = root / 'capture_full_summary.json'
index_path = root / 'capture_full_index.json'
report_hashes = {task['request_key']: hash_file(directory_for(task) / 'summary.json') for task in tasks}
inspection_hashes = {task['request_key']: hash_file(directory_for(task) / 'local_tensor_inspection.json') for task in tasks}
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    if (summary['request_key'] != batch_key or summary['report_sha256'] != report_hashes
            or summary['inspection_sha256'] != inspection_hashes or hash_file(index_path) != summary['index_sha256']):
        raise RuntimeError('Saved aggregate summary differs from the captured results.')
else:
    from statistics import fmean, median
    from collections import Counter
    index = {'request_key': batch_key, 'capture_manifest_key': capture_manifest['request_key'],
             'train': [], 'validation': [], 'policy': 'TRAIN for gradient updates; VALIDATION only for diagnostics and selection'}
    for task in tasks:
        report = reports[task['request_key']]
        row = report['row']
        index[row['split']].append({'capture_id': row['capture_id'], 'source_id': row['source_id'],
            'dimension': row['dimension'], 'request_key': task['request_key'],
            'tensor_path': (directory_for(task) / 'targets.pt').as_posix(), 'tensor_sha256': report['tensor_sha256'],
            'report_path': (directory_for(task) / 'summary.json').as_posix(),
            'records': report['captured_block_pair_records'], 'sampled_token_pairs': report['selected_token_pairs']})
    raw = (json.dumps(index, indent=2) + '\n').encode()
    if index_path.exists():
        if index_path.read_bytes() != raw:
            raise RuntimeError('Partial captured-target index differs.')
    else:
        with index_path.open('xb') as handle:
            handle.write(raw)
    groups = []
    for split in ('train', 'validation'):
        for layer in config['candidate_layers']:
            for stage in config['stages']:
                for role in ('target_image', 'prefix_text'):
                    values = [metric for report in reports.values() if report['row']['split'] == split
                              for metric in report['identity_metrics'] if metric['layer'] == layer
                              and metric['stage'] == stage['name'] and metric['role'] == role]
                    if not values:
                        continue
                    groups.append({'split': split, 'layer': layer, 'stage': stage['name'], 'role': role,
                        'n_prompts': len(values), 'mean_identity_relative_l2_error': fmean(v['identity_relative_l2_error'] for v in values),
                        'median_identity_relative_l2_error': median(v['identity_relative_l2_error'] for v in values),
                        'max_identity_relative_l2_error': max(v['identity_relative_l2_error'] for v in values),
                        'mean_identity_token_cosine': fmean(v['identity_mean_token_cosine'] for v in values),
                        'mean_identity_normalized_mse': fmean(v['identity_normalized_mse'] for v in values),
                        'sampled_tokens': sum(v['sampled_tokens'] for v in values)})
    summary = {'request_key': batch_key, 'status': 'passed', 'n_prompts': len(reports),
        'split_counts': dict(Counter(report['row']['split'] for report in reports.values())),
        'split_record_counts': {split: sum(report['captured_block_pair_records'] for report in reports.values() if report['row']['split'] == split) for split in ('train', 'validation')},
        'split_token_pair_counts': {split: sum(report['selected_token_pairs'] for report in reports.values() if report['row']['split'] == split) for split in ('train', 'validation')},
        'captured_block_pair_records': sum(report['captured_block_pair_records'] for report in reports.values()),
        'tensor_bytes': sum(report['tensor_bytes'] for report in reports.values()),
        'teacher_trajectory_calls': sum(report['teacher_trajectory_calls'] for report in reports.values()),
        'teacher_replay_calls': sum(report['teacher_replay_calls'] for report in reports.values()),
        'smoke_capture_reused': True, 'new_teacher_trajectories': 29, 'optimizer_updates': 0,
        'max_replay_relative_l2_error': max(control['relative_l2_error'] for report in reports.values() for control in report['replay_controls']),
        'max_replay_abs_error': max(control['max_abs_error'] for report in reports.values() for control in report['replay_controls']),
        'max_capture_peak_allocated_gib': max(report['peak_allocated_gib'] for report in reports.values()),
        'capture_seconds_sum': sum(report['capture_seconds'] for report in reports.values()),
        'groups': groups, 'report_sha256': report_hashes, 'inspection_sha256': inspection_hashes,
        'index_sha256': hashlib.sha256(raw).hexdigest(),
        'interpretation': 'unweighted per-prompt identity hidden-state diagnostics; no trained bridge or image-quality evidence'}
    with summary_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(summary, indent=2) + '\n')
print('Full capture pilot:', summary['status'], '| prompts:', summary['split_counts'])
print('Paired records:', summary['split_record_counts'], '| sampled token pairs:', summary['split_token_pair_counts'])
print('Saved tensor GiB:', summary['tensor_bytes'] / 2**30, '| maximum replay relative L2:', summary['max_replay_relative_l2_error'])
print('Teacher calls total:', summary['teacher_trajectory_calls'], '+', summary['teacher_replay_calls'], 'replay controls | smoke reused:', summary['smoke_capture_reused'])
for group in summary['groups']:
    if group['split'] == 'validation':
        print(f"Validation | block {group['layer']} | {group['stage']} | {group['role']} | mean identity L2: {group['mean_identity_relative_l2_error']:.6f} | mean cosine: {group['mean_identity_token_cosine']:.6f}")
print('Optimizer updates:', summary['optimizer_updates'], '| captured-target index:', index_path)


Reusing all 30 verified local captures; no Modal calls or teacher forwards.
Full capture pilot: passed | prompts: {'train': 20, 'validation': 10}
Paired records: {'train': 240, 'validation': 120} | sampled token pairs: {'train': 64280, 'validation': 31936}
Saved tensor GiB: 1.6237245853990316 | maximum replay relative L2: 0.0
Teacher calls total: 1200 + 90 replay controls | smoke reused: True
Validation | block 2 | early | target_image | mean identity L2: 0.081463 | mean cosine: 0.998280
Validation | block 2 | early | prefix_text | mean identity L2: 0.064367 | mean cosine: 0.993396
Validation | block 2 | middle | target_image | mean identity L2: 0.057803 | mean cosine: 0.998594
Validation | block 2 | late | target_image | mean identity L2: 0.084898 | mean cosine: 0.997557
Validation | block 4 | early | target_image | mean identity L2: 0.087636 | mean cosine: 0.996856
Validation | block 4 | early | prefix_text | mean identity L2: 0.118750 | mean cosine: 0.990890
Validation | block 4 | m

## What the complete capture pilot results mean

**All 30 frozen prompts now have verified teacher targets.** The first TRAIN capture was reused unchanged; **29 new trajectories** were captured using one shared loaded teacher. Every prompt has its own saved seed, resolution, prompt conditioning and fresh prefix cache. The frozen original teacher retains all 32 blocks. No bridge was inserted or updated during this capture batch.

| Split | Prompts | Paired block records | Sampled token pairs | Use |
|---|---:|---:|---:|---|
| TRAIN | 20 | 240 | 64,280 | Gradient updates when fitting begins |
| VALIDATION | 10 | 120 | 31,936 | Diagnostics and model selection only |
| Total | 30 | 360 | 96,216 | Complete implementation pilot |

The 360 records come from 30 prompts × 3 stages × 4 candidate blocks. Each block therefore has **60 TRAIN records and 30 VALIDATION records**, not 240 and 120 records of its own. A sampled token pair contains one teacher block input and its corresponding output. Each record samples 256 image tokens; early-stage records also contain between **16 and 57 valid prefix-text tokens** in this pilot. Cached stages contain no prefix queries. The same sampled indices are shared across all four candidates at each stage.

The saved targets retain the balanced pilot coverage: five capability groups, with four TRAIN and two VALIDATION prompts per group, 21 subdimensions, all five difficulty labels, and seven aspect ratios. This remains a small pilot. Token rows from one prompt are correlated and should not be treated as thousands of independent examples of prompt coverage.

**All 90 intact-teacher replay controls passed.** Across the full pilot, maximum relative L2 velocity difference is **0** and maximum absolute velocity difference is **0**. The tolerance remains 0.001 relative L2. Replay reconstructs the teacher prefix from the early saved call and uses the saved middle/late inputs. These checks support the consistency of captured latents, embeddings, masks, measured timesteps and cache handling. They compare teacher with teacher and are not learned-student results.

### Missing updates under identity on held-out validation prompts

Each table entry is the unweighted mean across the **10 validation prompts** of the relative L2 difference between sampled teacher block input and output. The denominator is that block's target output norm. Image and prefix text are evaluated separately. Prefix text is captured only at extraction, where it uses the zero-timestep conditioning path. None of these examples will be used for optimizer updates.

| Block | Early image error | Middle image error | Late image error | Prefix-text error |
|---|---:|---:|---:|---:|
| 2 | 8.15% | 5.78% | 8.49% | 6.44% |
| 3 | 10.32% | 7.81% | 8.44% | 6.57% |
| 4 | 8.76% | 9.77% | 10.01% | 11.88% |
| 5 | 10.33% | 10.48% | 10.63% | 48.99% |

These values quantify the residual update an identity-initialized adapter must learn. Lower identity error means a smaller missing local update for these sampled states, not necessarily an easier block to learn or better end-to-end image quality. Validation mean image-token cosine ranges from **0.995675 to 0.998594** across block/stage groups. High directional agreement can coexist with meaningful errors in update magnitude. We therefore retain both normalized-loss diagnostics and raw relative errors instead of using cosine alone to accept a replacement.

The TRAIN diagnostics use the same metric, with an unweighted mean across **20 training prompts**:

| Block | Early image error | Middle image error | Late image error | Prefix-text error |
|---|---:|---:|---:|---:|
| 2 | 8.03% | 5.66% | 8.61% | 6.69% |
| 3 | 10.37% | 7.92% | 8.69% | 6.68% |
| 4 | 8.63% | 9.89% | 10.54% | 11.85% |
| 5 | 10.24% | 10.61% | 10.37% | 48.80% |

TRAIN measurements describe the fitting data; VALIDATION measurements describe held-out data. Their means are not performance after training. The saved summary also records medians, maxima, normalized MSE, cosine and token counts by split, layer, stage and role. These support diagnosing large local errors and checking whether fitting behaves differently for prefix text and image tokens. They do not select a winning candidate; all four remain eligible for the same fitting budget.

Stage indices remain 0, 20 and 39 of a 40-step teacher trajectory. The measured middle-stage sigma ranges from **0.656238 to 0.657179** across the selected resolutions because the scheduler shifts its noise schedule with sequence length. Actual BF16 transformer input timesteps, scheduler values and replay latents are saved per prompt; stage labels must not be used as fixed noise values.

The 30 tensor bundles occupy **1.624 GiB** locally. This includes paired hidden states, shared prompt embeddings/masks, initial and stage latents, teacher and replay velocities, terminal latents, and tensor metadata/serialization. It differs from the earlier 1.52 GiB upper bound, which counted only paired hidden tensors and assumed up to 64 prefix tokens. Each bundle passed a local weights-only load, shape/finiteness checks, and shared-token-index checks; every file's checksum matches its remote capture report.

Across the saved pilot, **1200 teacher trajectory calls plus 90 replay calls** were executed. This total includes the existing smoke capture; this expansion added 29 × 43 = **1,247 calls**. The sum of recorded capture durations is **194.54 seconds**, excluding model loading, replay, serialization and downloading. The largest recorded allocated-memory peak during capture is **30.87 GiB**. Hooks and CPU copies affect these measurements, so they are capture diagnostics rather than a latency benchmark or training-memory estimate. No images were decoded.

**Decision supported:** this split-separated capture pilot is ready for bridge fitting. **Still uncertain:** convergence with the small prompt pool, whether three stages and sampled tokens are sufficient, whether local fitting preserves full-model velocity and image quality, and whether condition-image prompts behave similarly. We have **zero optimizer updates** and no trained bridge. Larger training pools and matched full-model evaluation remain necessary before accepting any replacement.

The split-separated target index is `data/phase5/capture_full_index.json`. Aggregate diagnostics and source hashes are in `capture_full_summary.json`; the immutable batch request is in `capture_full_request.json`. Per-prompt `targets.pt`, `summary.json`, `request.json`, and `local_tensor_inspection.json` remain under `data/phase5/teacher_targets/`. The remote results volume backs up every prompt's tensors, request and report. The first capture worker is preserved unchanged; `teacher_capture_batch_worker.py` adds reuse of the loaded pipeline for the expansion.

A completed rerun of all phase 4 and phase 5 code cells changed no saved artifact, imported no model or network client, and repeated no capture, replay, tensor inspection, selection or aggregate calculation. Initial bridge checkpoints and teacher weights remain unchanged.


## 4. Fit four independent bridge candidates to cached TRAIN targets

We train each saved rank-256 bridge for 500 AdamW updates, using only the 20 TRAIN prompts. The intact teacher is never loaded or called: its saved block inputs and outputs provide supervision. The same deterministic sample schedule is used for every candidate. Each 256-token batch contains 64 prompt tokens and 64 image tokens from each of the early, middle and late stages. We sample prompts uniformly, then tokens within a prompt, so longer prompts do not dominate fitting. All four groups contribute equally.

The loss is squared L2 distance between unit-normalized prediction and target, averaged over tokens, plus 0.1 times the per-token squared relative error in the unnormalized vectors. Summing normalized differences over the 4096 components avoids a tiny scale caused by feature averaging; it has the same directional minimum as normalized MSE. The magnitude term addresses the large prompt-token updates observed in the capture diagnostics. Learning rate is 0.001, weight decay 0.01 and gradient norm is clipped at 1.0. These are pilot choices, not tuned hyperparameters.

We evaluate TRAIN and VALIDATION before fitting and every 100 updates, without gradients. Checkpoint selection uses the lowest balanced objective on the 10 VALIDATION prompts, including the identity baseline. Every candidate still receives 500 updates. FP32 weights, residual additions and losses use BF16 autocast projections on a Modal L4; only bridge parameters are optimized. Prompt tokens are the saved hidden vectors representing the prompt, formerly labeled prefix text.

We retain checkpoints at initialization and every 100 updates, including model and optimizer states, random states, shared sampling-schedule hash, loss history, evaluations and the best weights. The selected checkpoint and complete results are backed up remotely and downloaded with checksum verification. Completed reruns reuse all results without training, evaluation or network calls. An interrupted attempt stops for inspection instead of automatically repeating unsaved updates. This is a small cached-state fitting experiment, not full-model quality validation.


In [4]:
# Fit four independent bridges to saved TRAIN targets, with cached validation and durable checkpoints.
import hashlib
import json
from pathlib import Path

root = Path('data/phase5')
index_path = root / 'capture_full_index.json'
index_raw = index_path.read_bytes()
index = json.loads(index_raw)
capture_summary_raw = (root / 'capture_full_summary.json').read_bytes()
capture_summary = json.loads(capture_summary_raw)
if capture_summary['status'] != 'passed' or hashlib.sha256(index_raw).hexdigest() != capture_summary['index_sha256']:
    raise RuntimeError('Captured target index differs or is incomplete.')
if len(index['train']) != 20 or len(index['validation']) != 10:
    raise RuntimeError('Fitting pilot split counts differ.')
if {record['source_id'] for record in index['train']} & {record['source_id'] for record in index['validation']}:
    raise RuntimeError('Training and validation source IDs overlap.')
prototype = json.loads(Path('data/phase4/bridge_prototypes_request.json').read_text(encoding='utf-8'))
initial_summary = json.loads(Path('data/phase4/bridge_prototypes_summary.json').read_text(encoding='utf-8'))
core_source = Path('data/phase4/bridge_definition.py').read_text(encoding='utf-8')
if hashlib.sha256(core_source.encode()).hexdigest() != prototype['request']['definition_sha256']:
    raise RuntimeError('Initial bridge implementation differs.')
capture_request = json.loads((root / 'capture_pilot_request.json').read_text(encoding='utf-8'))['request']
worker_path = root / 'bridge_fit_worker.py'
worker_source = worker_path.read_text(encoding='utf-8')

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

initial_payloads = {}
for layer in prototype['request']['candidate_layers']:
    path = Path(f'data/phase4/bridge_block_{layer:02d}_initial.pt')
    initial_payloads[str(layer)] = path.read_bytes()
    if hashlib.sha256(initial_payloads[str(layer)]).hexdigest() != initial_summary['checkpoint_sha256'][str(layer)]:
        raise RuntimeError('Initial prototype checkpoint changed.')
for split in ('train', 'validation'):
    for record in index[split]:
        if digest(Path(record['tensor_path'])) != record['tensor_sha256']:
            raise RuntimeError('Local teacher target bundle changed.')
request = {
    'version': 'phase5-bridge-fitting-pilot-v1', 'target_index': index,
    'target_index_sha256': hashlib.sha256(index_raw).hexdigest(),
    'capture_summary_sha256': hashlib.sha256(capture_summary_raw).hexdigest(),
    'prototype_request_key': prototype['request_key'], 'candidate_layers': prototype['request']['candidate_layers'],
    'initial_checkpoint_sha256': initial_summary['checkpoint_sha256'],
    'core_source': core_source, 'worker_source': worker_source,
    'worker_sha256': hashlib.sha256(worker_source.encode()).hexdigest(),
    'runtime': capture_request['runtime'], 'gpu': 'L4', 'seed': 20260928, 'sampling_seed': 20260929,
    'updates_per_bridge': 500, 'checkpoint_every': 100, 'tokens_per_group': 64,
    'groups': ['prompt_tokens', 'early_image', 'middle_image', 'late_image'],
    'sampling_policy': 'equal group weight; each token draws TRAIN prompt uniformly then token uniformly within that prompt; identical schedule for all candidates',
    'optimizer': 'AdamW', 'learning_rate': 0.001, 'weight_decay': 0.01, 'gradient_clip': 1.0,
    'magnitude_weight': 0.1,
    'objective': 'mean token squared L2 distance between unit-normalized prediction and target, plus 0.1 * mean per-token squared relative error; four groups weighted equally',
    'precision': 'FP32 parameters, residual addition and losses; BF16 autocast down/up projections',
    'evaluation': 'baseline and every 100 updates; unweighted mean over prompts within each group; equal group weights',
    'selection': 'minimum balanced validation objective at saved checkpoints including identity baseline; every candidate still receives 500 updates',
    'checkpoint_policy': 'retain initialization and every 100 updates with model, optimizer, RNG, schedule hash, history, evaluations and best state; interrupted run stops for inspection',
    'data_policy': 'only TRAIN gradients; VALIDATION no gradients, only diagnostics and checkpoint selection; no INTERNAL TEST or benchmark use',
    'scope': '20-prompt fitting implementation pilot; no full teacher/student forward or image evaluation',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = root / 'bridge_fitting_pilot'
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding='utf-8')) != manifest:
        raise RuntimeError('Fitting inputs or implementation changed; preserve trained results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_directory = f'/phase5/fitting/{key}'

def validate(report):
    if report['request_key'] != key:
        raise RuntimeError('Fitting report provenance differs.')
    if report['status'] != 'passed':
        raise RuntimeError('Saved fitting failed; preserve checkpoints, no automatic retry: ' + report.get('error', 'unknown'))
    if (report['total_optimizer_updates'] != 2000 or report['teacher_calls'] != 0
            or report['validation_optimizer_updates'] != 0 or len(report['results']) != 4):
        raise RuntimeError('Fitting result has unexpected counts or split use.')

report = None
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    validate(report)
    complete = True
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if not path.exists():
            complete = False
        elif digest(path) != artifact['sha256']:
            raise RuntimeError('Saved fitting checkpoint or result file changed.')
    if complete:
        print('Reusing verified local trained bridges, checkpoints and evaluations; no Modal call or optimizer update.')
    else:
        report = None
if report is None:
    import modal
    runtime = request['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        report = json.loads(b''.join(results_volume.read_file(remote_directory + '/summary.json')))
        print('Recovering saved remote fitting result; no GPU scheduled.')
    except FileNotFoundError:
        if report_path.exists() or any(run_dir.glob('*.pt')):
            raise RuntimeError('Partial local fitting has no remote final report; inspect instead of repeating updates.')
        app = modal.App('nihonga-phase5-bridge-fitting-pilot')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*runtime['packages'])
                 .env({'CUBLAS_WORKSPACE_CONFIG': ':4096:8', 'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'}))
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu=request['gpu'], serialized=True, timeout=1800,
            max_containers=1, retries=0, volumes={'/phase3': results_volume})(namespace['fit_bridges'])
        with modal.enable_output(), app.run():
            report = worker.remote(request, key, initial_payloads)
    if report['request_key'] != key:
        raise RuntimeError('Remote fitting report provenance differs.')
    if report['status'] == 'passed':
        for artifact in report['artifacts']:
            path = run_dir / artifact['name']
            if path.exists():
                if digest(path) != artifact['sha256']:
                    raise RuntimeError('Local checkpoint differs from remote backup.')
                continue
            temporary = path.with_suffix('.download')
            with temporary.open('wb') as handle:
                for chunk in results_volume.read_file(remote_directory + '/' + artifact['name']):
                    handle.write(chunk)
            if digest(temporary) != artifact['sha256']:
                raise RuntimeError('Downloaded fitting artifact checksum differs.')
            temporary.replace(path)
    if report_path.exists():
        if json.loads(report_path.read_text(encoding='utf-8')) != report:
            raise RuntimeError('Local fitting summary differs from backup.')
    else:
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(report, indent=2) + '\n')
    validate(report)

inspection_path = run_dir / 'local_checkpoint_inspection.json'
if inspection_path.exists():
    inspection = json.loads(inspection_path.read_text())
    if inspection['request_key'] != key or inspection['status'] != 'passed':
        raise RuntimeError('Saved checkpoint inspection differs or failed.')
else:
    import torch
    selected = []
    for result in report['results']:
        checkpoint = torch.load(run_dir / result['selected_checkpoint'], weights_only=True, map_location='cpu')
        if checkpoint['request_key'] != key or checkpoint['layer'] != result['layer'] or checkpoint['step'] != result['selected_step']:
            raise RuntimeError('Selected checkpoint metadata differs.')
        weights = checkpoint['state_dict']
        if weights['down.weight'].shape != (256, 4096) or weights['up.weight'].shape != (4096, 256):
            raise RuntimeError('Learned bridge shape differs.')
        if not all(torch.isfinite(tensor).all() for tensor in weights.values()):
            raise RuntimeError('Selected checkpoint contains nonfinite weights.')
        selected.append({'layer': result['layer'], 'step': result['selected_step'],
                         'nonzero_up_projection': bool(torch.count_nonzero(weights['up.weight']))})
    inspection = {'request_key': key, 'status': 'passed', 'weights_only_load': True, 'selected': selected}
    with inspection_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(inspection, indent=2) + '\n')
print('Bridge fitting:', report['status'], '| GPU:', report['gpu'], '| optimizer updates:', report['total_optimizer_updates'])
for result in report['results']:
    before = result['baseline_validation']['balanced_objective']
    after = result['selected_validation']['balanced_objective']
    print(f"Block {result['layer']} | selected step {result['selected_step']} | balanced validation objective {before:.6f} -> {after:.6f} | change {(after/before-1)*100:.2f}%")
    for initial, fitted in zip(result['baseline_validation']['groups'], result['selected_validation']['groups']):
        print(f"  {initial['group']} | validation relative L2 {initial['relative_l2_error']:.6f} -> {fitted['relative_l2_error']:.6f}")
print('Teacher calls:', report['teacher_calls'], '| validation optimizer updates:', report['validation_optimizer_updates'])
print('Selected checkpoints, full training states and reports:', run_dir)


Reusing verified local trained bridges, checkpoints and evaluations; no Modal call or optimizer update.
Bridge fitting: passed | GPU: NVIDIA L4 | optimizer updates: 2000
Block 2 | selected step 500 | balanced validation objective 0.007287 -> 0.003969 | change -45.53%
  prompt_tokens | validation relative L2 0.064367 -> 0.021131
  early_image | validation relative L2 0.081449 -> 0.048375
  middle_image | validation relative L2 0.057793 -> 0.041581
  late_image | validation relative L2 0.084883 -> 0.050429
Block 4 | selected step 400 | balanced validation objective 0.010011 -> 0.005036 | change -49.70%
  prompt_tokens | validation relative L2 0.118750 -> 0.020693
  early_image | validation relative L2 0.087622 -> 0.065940
  middle_image | validation relative L2 0.097682 -> 0.059122
  late_image | validation relative L2 0.100043 -> 0.054595
Block 5 | selected step 300 | balanced validation objective 0.017395 -> 0.006714 | change -61.41%
  prompt_tokens | validation relative L2 0.489928 ->

## What the first bridge-fitting results mean

**We completed 500 independent optimizer updates per candidate, for 2,000 updates in total.** Each adapter still has **2,097,152 parameters** and a 256-wide bottleneck. All fitting examples came from the 20 TRAIN prompts; the 10 VALIDATION prompts were used only for evaluation and checkpoint selection. The intact teacher was never loaded or run, and no validation gradient updates occurred.

The four candidates received the same deterministic sampling schedule. Each update uses 256 sampled tokens: 64 prompt tokens, then 64 image tokens from each of the early, middle and late denoising stages. Prompt tokens are the hidden vectors representing the prompt. They appear in the extraction-stage capture and affect downstream image attention through cached keys and values; they are not a separate kind of text dataset. Uniform prompt sampling prevents long prompts from receiving more weight just because they contain more tokens.

### Fitting objective and held-out validation

The objective averages four equally weighted groups. For each sampled token, it adds the squared distance between unit-normalized prediction and target to **0.1 times the squared unnormalized prediction error divided by the target's squared norm**. The first term compares direction; the second also penalizes differences in magnitude. The directional term sums over 4096 components before averaging across tokens, so its numerical scale differs from the feature-averaged normalized MSE in the capture reports. These objective values are loss units, not percentages, quality scores or direct velocity errors. Smaller is better.

| Block | Selected update | TRAIN objective: identity → selected | VALIDATION objective: identity → selected | Validation objective reduction |
|---|---:|---|---|---:|
| 2 | 500 | 0.007436 → 0.002093 | 0.007287 → 0.003969 | 45.53% |
| 3 | 300 | 0.008214 → 0.002835 | 0.008411 → 0.005215 | 37.99% |
| 4 | 400 | 0.009529 → 0.002762 | 0.010011 → 0.005036 | 49.70% |
| 5 | 300 | 0.015818 → 0.004057 | 0.017395 → 0.006714 | 61.41% |

Selection compares the identity baseline and the saved evaluations at updates 100, 200, 300, 400 and 500. It chooses the lowest validation objective, while every candidate receives the full 500-update budget. The TRAIN values describe fitting to seen prompts; VALIDATION values describe different held-out prompts. Because validation also chooses the checkpoint, these results are model-selection evidence, not an independent final test. Ten validation prompts provide limited coverage.

The selected checkpoints with nonzero learned up projections are blocks **2, 4, 5, 3**. These adapters can now make a nonzero residual update instead of behaving as identity. A nonzero update is not proof that it is useful in the complete model.

### What happened to the missing teacher updates

This table shows the mean relative L2 error across the 10 validation prompts within each group. It divides the predicted hidden-state error norm by the teacher output norm, exactly the kind of raw error that the identity diagnostics measured. It is reported separately from the combined fitting objective so changes in vector magnitude remain visible.

| Block | Validation group | Identity relative L2 error | Selected bridge relative L2 error |
|---|---|---:|---:|
| 2 | Prompt tokens | 6.44% | 2.11% |
| 2 | Early image tokens | 8.14% | 4.84% |
| 2 | Middle image tokens | 5.78% | 4.16% |
| 2 | Late image tokens | 8.49% | 5.04% |
| 3 | Prompt tokens | 6.57% | 2.25% |
| 3 | Early image tokens | 10.31% | 7.25% |
| 3 | Middle image tokens | 7.81% | 5.19% |
| 3 | Late image tokens | 8.43% | 5.24% |
| 4 | Prompt tokens | 11.87% | 2.07% |
| 4 | Early image tokens | 8.76% | 6.59% |
| 4 | Middle image tokens | 9.77% | 5.91% |
| 4 | Late image tokens | 10.00% | 5.46% |
| 5 | Prompt tokens | 48.99% | 2.31% |
| 5 | Early image tokens | 10.33% | 7.84% |
| 5 | Middle image tokens | 10.48% | 7.13% |
| 5 | Late image tokens | 10.63% | 6.53% |

Mean validation relative L2 error decreased in every reported block/token group. At the per-prompt level, raw relative error decreased in **160 of 160 paired block/group/prompt comparisons**. These are repeated measurements of the same 10 validation prompts, not 160 independent prompts or image-quality tests. Full-model predictions have not been evaluated.

For example, block 5's mean prompt-token error fell from **49.0% to 2.3%** of the teacher output norm. Its early image-token error fell from **10.3% to 7.8%**. The much stronger recovery on prompt tokens shows why the two roles must be reported separately; it does not imply that image behavior has recovered by the same amount.

The saved evaluations also contain directional loss, squared relative loss, cosine, largest per-prompt relative error and per-prompt values, separated by stage and token group. A lower mean can hide a regression on an individual prompt. The selected bridge has only seen cached states from the intact teacher. In an assembled student, surviving later blocks will receive modified states, which can accumulate differences absent from this isolated fitting experiment. No candidate has been accepted or jointly combined based on these results.

Training used AdamW with learning rate **0.001**, weight decay **0.01**, and gradient norm clipping at **1.0**. These are starting choices for this implementation pilot. The small prompt pool can be revisited many times; the number of sampled tokens does not turn 20 prompts into a large general-purpose training corpus. Longer or larger training has not been tested here.

Weights, residual additions and loss calculations used FP32, with BF16 autocast for the down/up projections. The report records GPU **NVIDIA L4** and Torch **2.14.0**. Fitting plus evaluations and checkpoint commits took **155.94 seconds**, with peak allocated GPU memory **0.39 GiB**. These are fitting diagnostics, not full-model generation benchmarks. The assembled BF16 pipeline also rounds the residual addition and weights differently, so production-dtype behavior remains to be checked.

**Decision supported:** the adapters can be fitted using cached teacher supervision while keeping TRAIN and VALIDATION separate. **Still uncertain:** whether these learned corrections preserve full-student velocity, prompt adherence and generated-image quality; whether prompt-token caches remain useful after insertion; and how the fitting scales beyond this 20-prompt pilot. No full student forward, generated image, benchmark or speedup measurement was produced in this step.

Selected weights are saved as `data/phase5/bridge_fitting_pilot/bridge_block_XX_selected.pt`. Initialization and updates 100–500 are retained in `bridge_block_XX_step_NNNN.pt`, including model/optimizer states, random states, history, evaluations and best weights. The shared sampling schedule, per-block progress and summaries, immutable request, aggregate summary and local checkpoint inspection are also saved. The checksummed fitting artifacts occupy **0.722 GiB**, excluding the local inspection, final report and existing teacher targets. All artifacts are backed up on the existing remote results volume; original phase 4 checkpoints remain unchanged.

Local weights-only inspection confirms selected checkpoint provenance, 4096↔256 projection shapes and finite weights. A completed rerun of every phase 4 and phase 5 code cell used the saved artifacts without model imports, teacher calls, training, validation forwards, checkpoint inspection, selection calculations or network calls, and changed no saved artifact. The per-checkpoint validation history remains available for review rather than being recomputed.
